# Phase 5 & 6: Train Baseline and FlexiModal MoE Models
Prepares sequential input window dataloaders, initializes model modules, and executes optimizer training runs.

In [ ]:
print("[BACKGROUND] Importing training and model modules...")
import os
import sys
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import warnings
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath('../'))
from src.models.baselines import EarlyFusionClassifier, GatedFusionClassifier, CrossAttentionFusionClassifier
from src.models.fleximodal_moe import FlexiModalMoE
from src.training.trainer import MultimodalTrainer
print("[STATUS] Deep learning imports verified successfully.")

In [ ]:
print("[BACKGROUND] Loading partitioned datasets directly...")
df_train = pd.read_csv('../data/processed/train/train_synced.csv')
df_val = pd.read_csv('../data/processed/val/val_synced.csv')

print(f"[INFO] Partitioned splits loaded. Train count: {df_train.shape[0]} | Val count: {df_val.shape[0]}")

In [ ]:
print("[BACKGROUND] Defining sequential PyTorch Dataset wrappers...")
class SyncedDataset(Dataset):
    def __init__(self, df, seq_len=5, is_train=False):
        self.df = df.reset_index(drop=True)
        self.seq_len = seq_len
        self.is_train = is_train
        # Locate feature columns dynamically supporting both real and mock columns
        face_cands = ['left_ear', 'right_ear', 'avg_ear', 'blink_velocity', 'brow_descent_left', 'brow_descent_right', 'brow_asymmetry', 'lip_compression', 'jaw_tension', 'mouth_corner_pull', 'forehead_tension', 'face_height_norm', 'head_tilt', 'temporal_x_var', 'temporal_y_var', 'eye_openness_ratio', 'landmark_confidence', 'nose_wrinkle']
        self.face_cols = [c for c in face_cands if c in df.columns]
        if len(self.face_cols) == 0:
            self.face_cols = [c for c in df.columns if c.startswith('face_') and c not in ['face_mask']]
            
        voice_cands = ['f0_mean', 'f0_std', 'f0_range', 'jitter_percent', 'shimmer_db', 'hnr', 'speaking_rate_proxy', 'voice_intensity', 'high_freq_ratio', 'spectral_flux', 'pause_ratio', 'voiced_fraction']
        self.voice_cols = [c for c in voice_cands if c in df.columns]
        if len(self.voice_cols) == 0:
            self.voice_cols = [c for c in df.columns if c.startswith('voice_') and c not in ['voice_mask']]
            
        physio_cands = ['ecg_rate_mean', 'ecg_hrv_rmssd', 'ecg_hrv_sdnn', 'eda_scl_mean', 'resp_rate_mean']
        self.physio_cols = [c for c in physio_cands if c in df.columns]
        if len(self.physio_cols) == 0:
            self.physio_cols = [c for c in df.columns if c.startswith('physio_') and c not in ['physio_mask']]
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        # Force conversion to float32 and handle NaNs dynamically from left join
        face_nans = self.df.loc[idx, self.face_cols].isna().any()
        voice_nans = self.df.loc[idx, self.voice_cols].isna().any()
        physio_nans = self.df.loc[idx, self.physio_cols].isna().any()
        
        face_mask = 0.0 if face_nans else 1.0
        voice_mask = 0.0 if voice_nans else 1.0
        physio_mask = 0.0 if physio_nans else 1.0
        
        # Randomly drop modalities during training to force model to learn actual features
        if self.is_train:
            if np.random.rand() < 0.3:
                face_mask = 0.0
            if np.random.rand() < 0.3:
                voice_mask = 0.0
            if np.random.rand() < 0.3:
                physio_mask = 0.0
            # Retain at least one active modality
            if face_mask == 0.0 and voice_mask == 0.0 and physio_mask == 0.0:
                physio_mask = 1.0
                
        face_mask_t = torch.tensor(face_mask, dtype=torch.float32)
        voice_mask_t = torch.tensor(voice_mask, dtype=torch.float32)
        physio_mask_t = torch.tensor(physio_mask, dtype=torch.float32)
        
        face_vals = np.nan_to_num(self.df.loc[idx, self.face_cols].values.astype(np.float32), nan=0.0)
        voice_vals = np.nan_to_num(self.df.loc[idx, self.voice_cols].values.astype(np.float32), nan=0.0)
        physio_vals = np.nan_to_num(self.df.loc[idx, self.physio_cols].values.astype(np.float32), nan=0.0)
        
        face_val = torch.tensor(face_vals, dtype=torch.float32).unsqueeze(0).repeat(self.seq_len, 1)
        voice_val = torch.tensor(voice_vals, dtype=torch.float32).unsqueeze(0).repeat(self.seq_len, 1)
        physio_val = torch.tensor(physio_vals, dtype=torch.float32).unsqueeze(0).repeat(self.seq_len, 1)
        label = torch.tensor(int(self.df.loc[idx, 'label']), dtype=torch.long)
        
        return {
            'face': face_val,
            'voice': voice_val,
            'physio': physio_val,
            'face_mask': face_mask_t,
            'voice_mask': voice_mask_t,
            'physio_mask': physio_mask_t,
            'label': label
        }

train_ds = SyncedDataset(df_train, is_train=True)
val_ds = SyncedDataset(df_val, is_train=False)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)
print("[STATUS] Dataloaders constructed successfully.")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Executing training cycles on target device: {device}")

# Construct a standard train loader (is_train=False) specifically for training baseline models
# that do not use training modality dropout.
train_ds_standard = SyncedDataset(df_train, is_train=False)
train_loader_standard = DataLoader(train_ds_standard, batch_size=128, shuffle=True)

print("[BACKGROUND] Training 1/5: Early Fusion baseline...")
model_early = EarlyFusionClassifier(face_dim=len(train_ds.face_cols), voice_dim=len(train_ds.voice_cols), physio_dim=len(train_ds.physio_cols))
opt_early = torch.optim.Adam(model_early.parameters(), lr=1e-3)
trainer_early = MultimodalTrainer(model_early, train_loader_standard, val_loader, opt_early, nn.CrossEntropyLoss(), device, '../outputs/checkpoints/best_early.pt', patience=5)
trainer_early.fit(num_epochs=10)

print("[BACKGROUND] Training 2/5: Gated Fusion baseline...")
model_gated = GatedFusionClassifier(face_dim=len(train_ds.face_cols), voice_dim=len(train_ds.voice_cols), physio_dim=len(train_ds.physio_cols))
opt_gated = torch.optim.Adam(model_gated.parameters(), lr=1e-3)
trainer_gated = MultimodalTrainer(model_gated, train_loader_standard, val_loader, opt_gated, nn.CrossEntropyLoss(), device, '../outputs/checkpoints/best_gated.pt', patience=5)
trainer_gated.fit(num_epochs=10)

print("[BACKGROUND] Training 3/5: Cross-Attention Fusion baseline...")
model_attn = CrossAttentionFusionClassifier(face_dim=len(train_ds.face_cols), voice_dim=len(train_ds.voice_cols), physio_dim=len(train_ds.physio_cols))
opt_attn = torch.optim.Adam(model_attn.parameters(), lr=1e-3)
trainer_attn = MultimodalTrainer(model_attn, train_loader_standard, val_loader, opt_attn, nn.CrossEntropyLoss(), device, '../outputs/checkpoints/best_attention.pt', patience=5)
trainer_attn.fit(num_epochs=10)

print("[BACKGROUND] Training 4/5: Standard MoE Fusion (without modality dropout)...")
model_moe_standard = FlexiModalMoE(face_dim=len(train_ds.face_cols), voice_dim=len(train_ds.voice_cols), physio_dim=len(train_ds.physio_cols), num_experts=3)
opt_moe_std = torch.optim.Adam(model_moe_standard.parameters(), lr=1e-3)
trainer_moe_std = MultimodalTrainer(model_moe_standard, train_loader_standard, val_loader, opt_moe_std, nn.CrossEntropyLoss(), device, '../outputs/checkpoints/best_moe_standard.pt', patience=5)
trainer_moe_std.fit(num_epochs=10)

print("[BACKGROUND] Training 5/5: Robust FlexiModal MoE (with training modality dropout)...")
model_moe_robust = FlexiModalMoE(face_dim=len(train_ds.face_cols), voice_dim=len(train_ds.voice_cols), physio_dim=len(train_ds.physio_cols), num_experts=3)
opt_moe_rob = torch.optim.Adam(model_moe_robust.parameters(), lr=1e-3)
trainer_moe_rob = MultimodalTrainer(model_moe_robust, train_loader, val_loader, opt_moe_rob, nn.CrossEntropyLoss(), device, '../outputs/checkpoints/best_moe_robust.pt', patience=5)
trainer_moe_rob.fit(num_epochs=10)

print("[STATUS] All 5 models successfully trained and checkpoints saved under early_fusion/outputs/checkpoints/")